# India Traffic Rules RAG Pipeline

Single notebook covering the full pipeline: document extraction summary, OCR-aware loading, chunking, embedding, vector storage, retrieval, and generation.

Follows the library/code conventions from `reference/` (LangChain + Groq + Ollama + FAISS/Chroma).

## Phase 1: Setup

In [1]:
import os, json, re, getpass, warnings
import numpy as np
from dotenv import load_dotenv
from uuid import uuid4
from IPython.display import display, Markdown

In [2]:
warnings.filterwarnings('ignore')
load_dotenv(override=True)

True

In [3]:
#Check for Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [4]:
#Set Hugging Face token (needed only if you later use a gated HF model - the
#cross-encoder reranker used in this notebook is public and doesn't require it)
if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"]:
    os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace Token: ")

#HfFolder.save_token was removed in huggingface_hub v1.x - login() is the current
#equivalent, but (unlike the old call) it validates the token against the HF API,
#so a placeholder/missing token shouldn't block the rest of the notebook.
from huggingface_hub import login
try:
    login(token=os.environ["HF_TOKEN"])
except Exception as e:
    print(f"HF login skipped ({e}) - fine unless a gated model is needed later.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Windows system dependencies (Poppler + Tesseract)

The reference notebooks install Poppler/Tesseract via `brew` (Mac). On Windows we use Tesseract's official installer (via `winget`) and a portable Poppler build extracted into `tools/` (no admin rights needed). This cell just puts both on `PATH` for this kernel process, since a fresh Jupyter kernel won't automatically pick up a PATH change made after it started.

In [5]:
#Machine-specific paths - adjust if this project is moved to another machine
TESSERACT_DIR = r"C:\Program Files\Tesseract-OCR"
POPPLER_DIR = r"C:\Dhiren\Jio Institute\PROJECTS\GenAI\tools\poppler-26.09.0\Library\bin"
TESSDATA_DIR = r"C:\Dhiren\Jio Institute\PROJECTS\GenAI\tools\tessdata"  # extra language packs (Program Files\Tesseract-OCR only ships English)

os.environ["PATH"] = TESSERACT_DIR + os.pathsep + POPPLER_DIR + os.pathsep + os.environ.get("PATH", "")
os.environ["TESSDATA_PREFIX"] = TESSDATA_DIR

## Phase 2: Document Extraction (Data Collection)

Documents were collected with `scripts/extract_documents.py`, which walks the verified government-source inventory in `research/india_traffic_document_links.xlsx` and downloads every reachable PDF into `data/raw/full_extraction/<category>/`, recording full provenance (source, page URL, file URL, SHA-256, timestamp) in `data/raw/full_extraction/manifest.csv`.

That's a scraping/ETL concern, not a RAG-pipeline concern, so it stays as a standalone script rather than notebook cells - this section just loads the resulting manifest so the corpus is visible here.

In [6]:
import csv
from pathlib import Path
from collections import Counter

CORPUS_DIR = Path("../data/raw/full_extraction")
MANIFEST_PATH = CORPUS_DIR / "manifest.csv"

pdf_files = list(CORPUS_DIR.rglob("*.pdf"))
print(f"Total PDFs on disk: {len(pdf_files)}")

by_category = Counter(f.parent.name for f in pdf_files)
for category, count in sorted(by_category.items(), key=lambda x: -x[1]):
    print(f"  {category:35s} {count}")

Total PDFs on disk: 1424
  union-territory                     478
  central-statutory-technical-body    452
  state                               432
  state-rta-sta                       33
  state-highway-traffic-police        15
  state-road-safety-authority-cell    14


In [7]:
with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

print(f"Manifest entries: {len(manifest_rows)}")

errors = Counter(r["error"] for r in manifest_rows if r["error"] and r["error"] != "skipped_existing")
print(f"\nFailed downloads by reason ({sum(errors.values())} total):")
for reason, count in errors.most_common(10):
    print(f"  {count:4d}  {reason}")

Manifest entries: 1174

Failed downloads by reason (71 total):
    16  http_404
    11  no_pdf_links_found_on_page (may be JS-rendered - needs manual review)
     2  http_500
     1  request_error: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Read timed out. (read timeout=20)
     1  request_error: ('Connection broken: IncompleteRead(3445970 bytes read, 91201132 more expected)', IncompleteRead(3445970 bytes read, 91201132 more expected))
     1  request_error: ('Connection broken: IncompleteRead(1913742 bytes read, 6871226 more expected)', IncompleteRead(1913742 bytes read, 6871226 more expected))
     1  request_error: ('Connection broken: IncompleteRead(3617280 bytes read, 118580701 more expected)', IncompleteRead(3617280 bytes read, 118580701 more expected))
     1  request_error: HTTPSConnectionPool(host='jhtransport.gov.in', port=443): Max retries exceeded with url: /acts-rules.html (Caused by ConnectTimeoutError(<HTTPSConnection(host='jhtransport.gov.in', port=443)

## Phase 3: Document Loading (OCR-aware)

A sample check earlier found that roughly half the downloaded PDFs are scanned/photocopied government documents with no embedded text layer - a plain text extractor would return nothing for those. `UnstructuredPDFLoader` with `strategy='hi_res'` runs layout analysis and falls back to OCR (via the `OCR_AGENT` set in `.env`, Tesseract) automatically, so the same loader call handles both text-native and scanned PDFs.

`hi_res` is slow per file (layout model + possible OCR), so this is demonstrated on a small sample first before scaling to the full corpus.

In [8]:
from langchain_community.document_loaders import UnstructuredPDFLoader

In [9]:
#Just the Gujarati file for this validation pass - it's the one that actually matters here
#(real non-English test case for OCR + translation). A second file (26-page, table-heavy BPRD
#doc) caused unpredictable timeouts - each nbconvert run starts a fresh kernel, so the
#layout/table models get loaded from scratch every time, and table-structure inference on a
#table-heavy document adds highly variable cost on top of that. Broader multi-file validation
#happens once the full-corpus script exists (see the timing note below).
gujarati_candidates = [f for f in pdf_files if "gujarat" in f.name.lower() and "gujarati" in f.name.lower()]
sample_files = [gujarati_candidates[0]]

print(f"Sample: {len(sample_files)} file")
for f in sample_files:
    print(f"  [{f.parent.name}] {f.name}")

Sample: 1 file
  [state-road-safety-authority-cell] gujarat-road-safety-authority-gujrosa-ac__gujarat-right-to-information-rules-2010-gujarati-2-113640b9.pdf


In [10]:
#Timing reality check (measured directly, outside this notebook): hi_res on an 11-page file took
#~103s with 3 languages (~9.4s/page) vs ~79s with English only - so per-page cost is dominated by
#the hi_res layout model itself, not language count. Across the full corpus (~20-25k pages) that's
#50+ hours single-threaded, ~6-7 hours even parallelized across all 8 cores. Full-corpus OCR will
#need a separate parallelized, resumable script (same pattern as scripts/extract_documents.py's
#manifest-based resume) - not something to run synchronously in a notebook cell.
#
#Language list is trimmed for this demo: combining all 13 installed languages made Tesseract's
#combined recognition search space slow enough to time out (see the failed run above). eng+hin+guj
#is the exact combo already timed at ~103s/11 pages - it covers English, Hindi (Devanagari script)
#and Gujarati (a distinct script - Hindi's Devanagari model can't read it), matching our sample.
#The full-scale script will need cheap per-file script/language detection (e.g. Tesseract OSD) to
#pick the right 1-2 languages per file instead of guessing or brute-forcing every language.
INDIAN_LANGUAGES = ["eng", "hin", "guj"]

sample_docs = {}
for f in sample_files:
    loader = UnstructuredPDFLoader(str(f),
                                   strategy='hi_res',
                                   languages=INDIAN_LANGUAGES,
                                   extract_images_in_pdf=False,
                                   infer_table_structure=True,
                                   chunking_strategy="by_title", # section-based chunking
                                   max_characters=4000, # max size of chunks
                                   new_after_n_chars=3800, # preferred size of chunks
                                   combine_text_under_n_chars=2000, # smaller chunks < 2000 chars will be combined into a larger chunk
                                   mode='elements')
    sample_docs[f.name] = loader.load()
    print(f"[{f.parent.name}] {f.name}: {len(sample_docs[f.name])} elements")

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[state-road-safety-authority-cell] gujarat-road-safety-authority-gujrosa-ac__gujarat-right-to-information-rules-2010-gujarati-2-113640b9.pdf: 13 elements


In [11]:
#Inspect one sample's elements and content
sample_name = sample_files[0].name
print([doc.metadata.get('category') for doc in sample_docs[sample_name]])
print("\n--- First element content ---\n")
print(sample_docs[sample_name][0].page_content)

['CompositeElement', 'CompositeElement', 'CompositeElement', 'CompositeElement', 'Table', 'CompositeElement', 'Table', 'CompositeElement', 'Table', 'CompositeElement', 'Table', 'CompositeElement', 'CompositeElement']

--- First element content ---

કિ ત્રસ...”

Extra No. 61

REGISTERED NO. G/GNR/2

EXTRAORDINARY PUBLISHED BY AUTHORITY

Vol. LI]

MONDAY, APRIL 26, 2010NAISAKHA 6, 1932

Separate paging is given to this part in order that it may be filed as a Separate Compilation.

PART IV-A

Rules and Orders (Other than those published in Part I, I-A and I-L) made by the Government of Gujarat under the Central Acts.

सामान्य વહીવટ વિભાગ,

જાહેરનામું

સચિવાલય,ગાંધીનગર, २२भी માર્ચ, २०१०.

માહિતીનો અધિકાર અધિનિયમ, ૨૦૦૫.

ક્રમાંકઃ જીએસ | ૨૧ / २०१० | વીએચએસ | ૧૦૦૫ / २४३४४ / આર.ટી.આઇ.સેલઃ- માહિતીનોઅધિકાર અધિનિયમ, ૨૦૦૫ (સન ૨૦૦૫ના રરમા)ની કલમ-૨૭ થી મળેલી સત્તાની રૂએ અને આ અર્થે કરેલા વિઘમાન નિયમો રદ કરીને, ગુજરાત સરકાર, આથી, નીચેના નિયમો કરે છે:-

<?a till w.4 w1Xw:

P.

(9) YL kqql %V?Ld ~lkddl

### Translate non-English content to English

Everything gets stored in English regardless of source language. OCR above reads the document in its *actual* script/language (that's what the multi-language `languages=` list was for - translating garbled English-OCR of Gujarati/Hindi/etc. text would just produce garbled English). This step detects the language of each extracted element with `langdetect` and, for anything not already English, translates it using the same Groq chat model pattern as the Generation phase.

In [12]:
from langdetect import detect, LangDetectException
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate  # langchain.prompts was removed in langchain 1.x

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq")

translate_template = """Translate the following text to English. Preserve all factual content,
numbers, names, section/clause references, and formatting exactly - this is an official government
document and accuracy matters more than fluency. If the text is already in English, return it unchanged.
Return ONLY the translated text, no commentary.

Text:
{text}
"""
translate_prompt = ChatPromptTemplate.from_template(translate_template)
translate_chain = translate_prompt | llm

In [13]:
def detect_language(text: str) -> str:
    """ISO 639-1 code, or 'unknown' if the text is too short/ambiguous to detect reliably."""
    text = text.strip()
    if len(text) < 10:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def translate_to_english(text: str) -> str:
    return translate_chain.invoke({"text": text}).content

In [14]:
#Test on the Gujarati sample - the clearest non-English case in this sample set
gujarat_file = [f for f in sample_files if "gujarat" in f.name.lower()][0]
gujarat_elements = sample_docs[gujarat_file.name]

for i, doc in enumerate(gujarat_elements[:5]):
    lang = detect_language(doc.page_content)
    print(f"--- Element {i} (detected: {lang}) ---")
    print("Original:", doc.page_content[:200])
    if lang not in ("en", "unknown"):
        translated = translate_to_english(doc.page_content)
        print("Translated:", translated[:200])
    print()

--- Element 0 (detected: gu) ---
Original: કિ ત્રસ...”

Extra No. 61

REGISTERED NO. G/GNR/2

EXTRAORDINARY PUBLISHED BY AUTHORITY

Vol. LI]

MONDAY, APRIL 26, 2010NAISAKHA 6, 1932

Separate paging is given to this part in order that it may be


Translated: What fear...”

Extra No. 61

REGISTERED NO. G/GNR/2

EXTRAORDINARY PUBLISHED BY AUTHORITY

Vol. LI]

MONDAY, APRIL 26, 2010 NAISAKHA 6, 1932

Separate paging is given to this part in order that it may

--- Element 1 (detected: gu) ---
Original: અરજીનો નિકાલઃ

(1) %b? rlkd wR51?74 w ? ~ t t $ ri4d r~kd, A 4 hid rw.14 da Rqa4 Gt? ~7fl V L S ~ %6$1 welql whRwiri RE% 3 2 ~ ~ ZLFJLLQL SL?I?L~? l i d s t ? 5 s 4 %d& welq~. ri4c4 r~kd wiRllr UYLI?L


Translated: Application Disposal:

(1) %b? rlkd wR51?74 w ? ~ t t $ ri4d r~kd, A 4 hid rw.14 da Rqa4 Gt? ~7fl V L S ~ %6$1 welql whRwiri RE% 3 2 ~ ~ ZLFJLLQL SL?I?L~? l i d s t ? 5 s 4 %d& welq~. ri4c4 r~kd wiRll

--- Element 2 (detected: gu) ---
Original: $. અપીલઃ

(૧) જાહેર માહિતી અધિકારીના નિર્ણયથી નારાજ થયેલી અથવા અધિનિયમની ક્લમ--૭ માં ઠરાવેલી સમયમર્યાદામાં જેને કોઇ નિર્ણય ન મળ્યો હોય તેવી કોઇપણ વ્યક્તિ, તેને એવો નિર્ણય મળ્યાની અથવા ન મળ્યાની તારીખથ


Translated: $. Appeal:

(1) Any person who is dissatisfied with the decision of the Public Information Officer or who has not received any decision within the time limit prescribed in clause‑7 of the Act may, wit

--- Element 3 (detected: gu) ---
Original: ના્ગરેક્તાની ઓળખ:

અધિનિયમની કલમ-# હેઠળ અરજી મળ્યે, જાહેર માહિતી અધિકારીનો એવો અભિપ્રાય હોય કે નાગરિક્તાની ઓળખ સુનિશ્ચિત કરવાનું જરૂરી છે ત્યારે, તે, અધિનિયમની ક્લમ-૭ હેઠળ ઠરાવેલી સમયમર્યાદાના પાલનને 


Translated: Identification of citizenship:

When an application is received under Section-# of the Act, if the Public Information Officer is of the opinion that it is necessary to ascertain the identity of the ci

--- Element 4 (detected: gu) ---
Original: બેંક / પોસ્ટ કોની તરફેણમાં રકમ ઓફિસ નું નામ અને સ્થળ ડિમાન્ડ ડ્રાફટ | पे | તારીખ ઓડર / ભારતીય પોસ્ટલ ઓડરનો નંબર રૂપિયા *(૧) “ગુજરાત સરકાર' CC તકક (સરકારી વિભાગ/ કચેરી સિવાયના જાહેર સત્તામંડળનું નામ નિ


Translated: Bank / Post In favor of Amount Office name and location Demand Draft | Pay | Date Order / Indian Postal Order number Rupees *(1) “Gujarat Government” CC Takak (Specify name of public authority other t



## Scope for this build: curated text-native subset

Time-boxed decision: full-corpus OCR (~24k pages, 50+ hours) and even the full text-native set
(84 files, ~69 min embedding alone) don't fit tonight's deadline. This is a fixed, curated list
of 16 text-native files (no OCR needed) covering the Central MV Act/Rules plus core Acts/Rules
from Gujarat, Delhi, Mizoram, Himachal Pradesh, Andaman & Nicobar, and CPCB emission standards -
~1,150 pages, ~23 min to embed. Everything else stays on disk in `data/raw/full_extraction/` for
a future pass.

In [15]:
SCOPE_FILES = [
    r"..\data\raw\full_extraction\state\madhya-pradesh-mv-rules-1994-direct-pdf__a1988-59-b740e529.pdf",  # Central MV Act 1988
    r"..\data\raw\full_extraction\union-territory\andaman-nicobar-islands-transport-acts-r__the-central-motor-vehicles-rules-1989-14ddf947.pdf",  # Central MV Rules 1989
    r"..\data\raw\full_extraction\state\gujarat-commissionerate-of-transport-act__gmv-rules-1989-a99d3b52.pdf",
    r"..\data\raw\full_extraction\state\gujarat-commissionerate-of-transport-act__bmv-taact-1958-ba51970a.pdf",
    r"..\data\raw\full_extraction\state\gujarat-commissionerate-of-transport-act__bmv-tax-rule-1959-cb64db21.pdf",
    r"..\data\raw\full_extraction\union-territory\delhi-nct-transport-acts-and-rules-notif__delhi-motor-vehicles-rules-1993-as-amended-secreta-52c5b4c7.pdf",
    r"..\data\raw\full_extraction\union-territory\delhi-nct-transport-acts-and-rules-notif__the-delhi-motor-vehicle-taxation-act-r-2af515c1.pdf",
    r"..\data\raw\full_extraction\union-territory\delhi-nct-transport-acts-and-rules-notif__mact-6a89af46.pdf",
    r"..\data\raw\full_extraction\state\mizoram-act-rules__the-mizoram-motor-vehicle-taxation-act1996-4d1a51ab.pdf",
    r"..\data\raw\full_extraction\state\mizoram-act-rules__online-special-road-permit-dil-dan-kimchang-1-8ac37036.pdf",
    r"..\data\raw\full_extraction\state\mizoram-act-rules__sop-for-vlteas-1893dd12.pdf",
    r"..\data\raw\full_extraction\state-road-safety-authority-cell\himachal-pradesh-road-safety-cell__action-plan-2024-25-beaf0f7b.pdf",
    r"..\data\raw\full_extraction\union-territory\andaman-nicobar-islands-transport-acts-r__an-motor-aggregator-scheme-dca46278.pdf",
    r"..\data\raw\full_extraction\union-territory\andaman-nicobar-islands-transport-acts-r__finedet-e27a23fe.pdf",
    r"..\data\raw\full_extraction\central-statutory-technical-body\cpcb-vehicular-exhaust-emission-standard__it-technical-activity-2108d906.pdf",
] + [str(p) for p in Path(r"..\data\raw\full_extraction\state").glob("haryana-hartrans-gov-in-acts-rules__*.pdf")]

SCOPE_FILES = [Path(f) for f in SCOPE_FILES]
print(f"{len(SCOPE_FILES)} files in scope")
for f in SCOPE_FILES:
    exists = f.exists()
    print(f"  {'OK ' if exists else 'MISSING'} {f.name}")

24 files in scope
  OK  madhya-pradesh-mv-rules-1994-direct-pdf__a1988-59-b740e529.pdf
  OK  andaman-nicobar-islands-transport-acts-r__the-central-motor-vehicles-rules-1989-14ddf947.pdf
  OK  gujarat-commissionerate-of-transport-act__gmv-rules-1989-a99d3b52.pdf
  OK  gujarat-commissionerate-of-transport-act__bmv-taact-1958-ba51970a.pdf
  OK  gujarat-commissionerate-of-transport-act__bmv-tax-rule-1959-cb64db21.pdf
  OK  delhi-nct-transport-acts-and-rules-notif__delhi-motor-vehicles-rules-1993-as-amended-secreta-52c5b4c7.pdf
  OK  delhi-nct-transport-acts-and-rules-notif__the-delhi-motor-vehicle-taxation-act-r-2af515c1.pdf
  OK  delhi-nct-transport-acts-and-rules-notif__mact-6a89af46.pdf
  OK  mizoram-act-rules__the-mizoram-motor-vehicle-taxation-act1996-4d1a51ab.pdf
  OK  mizoram-act-rules__online-special-road-permit-dil-dan-kimchang-1-8ac37036.pdf
  OK  mizoram-act-rules__sop-for-vlteas-1893dd12.pdf
  OK  himachal-pradesh-road-safety-cell__action-plan-2024-25-beaf0f7b.pdf
  OK  andaman

In [16]:
#Fast strategy (no OCR needed - these are all text-native) - loads whole-document text per file
all_docs = []
for f in SCOPE_FILES:
    loader = UnstructuredPDFLoader(str(f), strategy="fast", mode="single")
    docs = loader.load()
    for d in docs:
        d.metadata["source_file"] = f.name
    all_docs.extend(docs)
    print(f"loaded: {f.name} ({len(docs[0].page_content)} chars)")

print(f"\nTotal documents: {len(all_docs)}")

No languages specified, defaulting to English.


loaded: madhya-pradesh-mv-rules-1994-direct-pdf__a1988-59-b740e529.pdf (369727 chars)


No languages specified, defaulting to English.


loaded: andaman-nicobar-islands-transport-acts-r__the-central-motor-vehicles-rules-1989-14ddf947.pdf (380116 chars)


No languages specified, defaulting to English.


...

No languages specified, defaulting to English.


loaded: haryana-hartrans-gov-in-acts-rules__2022071682-febb07a9.pdf (27981 chars)


No languages specified, defaulting to English.


No languages specified, defaulting to English.


loaded: haryana-hartrans-gov-in-acts-rules__2022080897-71ca4bc7.pdf (10711 chars)
loaded: haryana-hartrans-gov-in-acts-rules__2022103186-bf1e811b.pdf (0 chars)

Total documents: 24


## Phase 4: Splitting Documents

Same `RecursiveCharacterTextSplitter` pattern as the reference notebooks.

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters) - hyperparameter that can be tuned
    chunk_overlap=200,  # chunk overlap (characters) - hyperparameter that can be tuned
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(all_docs)

print(f"Split {len(all_docs)} documents into {len(all_splits)} chunks.")

Split 24 documents into 3235 chunks.


## Phase 5: Embedding + Vector Store

Local Ollama embeddings (`nomic-embed-text`, ~3.4 chunks/sec measured on this machine) into a
persisted Chroma store, matching the reference notebooks' pattern.

In [18]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

vector_store = Chroma(
    collection_name="india-traffic-rules",
    embedding_function=embeddings_model,
    persist_directory="../chroma_langchain_db",
)

In [19]:
#Batched (not one giant call) - a single request with thousands of texts was hitting an
#intermittent Ollama internal connection error (its tokenize helper refusing the connection
#on a random port). Small batches are also resumable if one batch fails.
BATCH_SIZE = 50
uuids = [str(uuid4()) for _ in range(len(all_splits))]

for i in range(0, len(all_splits), BATCH_SIZE):
    batch_docs = all_splits[i:i + BATCH_SIZE]
    batch_ids = uuids[i:i + BATCH_SIZE]
    for attempt in range(3):
        try:
            vector_store.add_documents(documents=batch_docs, ids=batch_ids)
            break
        except Exception as e:
            print(f"  batch {i}-{i+len(batch_docs)} attempt {attempt+1} failed: {e}")
            time.sleep(2)
    else:
        print(f"  batch {i}-{i+len(batch_docs)} FAILED after 3 attempts - skipped")
    if i % 500 == 0:
        print(f"embedded {i}/{len(all_splits)}")

print(f"Done. Added {len(all_splits)} chunks to the vector store.")

embedded 0/3235


embedded 500/3235


embedded 1000/3235


embedded 1500/3235


embedded 2000/3235


embedded 2500/3235


embedded 3000/3235


Done. Added 3235 chunks to the vector store.


## Phase 6: Retrieval and Generation

Strict prompt: answer only from retrieved context, explicitly refuse otherwise - per this
project's original requirement that the chatbot never invents traffic-law content.

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

strict_template = """You are a traffic-rules assistant for India. Answer the question using ONLY
the context below, which comes from official government Acts, Rules and regulations. Cite the
source document (from the "Source:" tag on each context chunk) in your answer.

If the context does not contain enough information to answer, say clearly that this specific
knowledge base does not cover it - do NOT use outside knowledge or guess.

Context:
{context}

Question: {question}
"""
strict_prompt = ChatPromptTemplate.from_template(strict_template)

def format_docs(docs):
    return "\n\n".join(f"Source: {d.metadata.get('source_file', 'unknown')}\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | strict_prompt
    | llm
    | StrOutputParser()
)

In [21]:
#In-scope question (covered by our corpus)
response = rag_chain.invoke("What are the rules for motor vehicle aggregators in Andaman and Nicobar Islands?")
display(Markdown(response))

**Motor Vehicle Aggregator Scheme – 2024 (Andaman & Nicobar Islands)**  

The draft scheme issued by the Directorate of Transport, Andaman & Nicobar Administration sets out the following rules for any person or entity that wishes to operate as a **motor‑vehicle aggregator** in the Union Territory:

| Rule / Requirement | What the scheme says |
|--------------------|----------------------|
| **Short title, extent and commencement** | The scheme is called the *Motor Vehicle Aggregator Scheme, 2024* and comes into force on the date of its publication in the Official Gazette. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, Section 1) |
| **Definitions** | The scheme defines an “aggregator” as an association of drivers or motor‑vehicle owners, or any other association or a Limited Liability Partnership formed under the LLP Act, 2008. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, Section 2) |
| **Registered office / branch** | The applicant must have a **registered office in India**. If the State requires, the aggregator may also establish a **branch office within the State** where it intends to operate. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, point 2) |
| **Compliance with other statutes** | The aggregator must comply with all applicable laws, **including**: <br>• Motor Vehicles Act, 1988 (and its rules) <br>• Digital Personal Data Protection Act, 2023 <br>• Consumer Protection Act, 2019 <br>• Information Technology Act, 2000 (and related intermediary guidelines). (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, point 3) |
| **License prerequisite for driver integration** | An applicant **shall not integrate any driver or represent himself as an aggregator unless he holds a valid licence** issued for that purpose. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, point 4) |
| **Conditions for grant of licence** | To obtain a licence, the applicant must **demonstrate compliance with the details of vehicles and drivers integrated with them**. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, “Condition for grant of License for Aggregator”) |
| **Confidentiality** | The State Government shall keep **all documents and information obtained from the aggregator confidential** and secret. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, point 4) |
| **Application form (Form‑I) – required particulars** | The licence application must include: <br>1. Full name of the applicant <br>2. Address of the main office <br>3. Number and addresses of any branches in the Islands <br>4. If a company – certificate of incorporation & memorandum of association; if a firm – certificate of registration <br>5. Name, contact details of key managerial personnel or authorised signatory <br>6. Telephone, website and email‑ID <br>7. Number and type of vehicles to be operated, together with a separate list of vehicle numbers and permit particulars <br>8. Details of GPS/GPRS facility <br>9. Details of other infrastructure <br>10. Details of returns filed in the last three years <br>11. Copies of financial statements for the last three years <br>12. Details of fee paid. (Source: *andaman‑nicobar‑islands‑transport‑acts‑r__an‑motor‑aggregator‑scheme‑dca46278.pdf*, “FORM‑I Application for the Grant of License for Aggregator”) |

**In summary**, the scheme establishes a formal licensing process for aggregators, mandates a registered (and possibly branch) office, requires strict compliance with the Motor Vehicles Act and several consumer‑data‑protection statutes, prohibits driver integration without a licence, obliges the applicant to furnish detailed corporate, vehicle and financial information, and guarantees confidentiality of the submitted documents. The scheme becomes effective once published in the Official Gazette.

In [22]:
#Out-of-scope question (Kerala isn't in this curated corpus) - should refuse, not guess
response = rag_chain.invoke("What is the parking fine in Kerala?")
display(Markdown(response))

The documents you have supplied contain provisions relating to penalties for obstructing traffic in Madhya Pradesh and rules on fare meters and vehicle safety in Delhi, but they do **not** include any information about parking fines in the state of Kerala.  

**Therefore, this specific knowledge base does not cover the parking fine applicable in Kerala.**